In [ ]:
## Авторство
## Данный проект разработан и принадлежит ООО «Карпов Курсы».  
## Официальный сайт: https://karpov.courses

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sqlalchemy import create_engine
engine = create_engine("")

In [ ]:
user_data = pd.read_sql('SELECT * FROM public.user_data;', con=engine) 

,user_id,gender,age,country,city,exp_group,os,source
0,200,1,34,Russia,Degtyarsk,3,Android,ads
1,201,0,37,Russia,Abakan,0,Android,ads
2,202,1,17,Russia,Smolensk,4,Android,ads
3,203,0,18,Russia,Moscow,1,iOS,ads
4,204,0,36,Russia,Anzhero-Sudzhensk,3,Android,ads


In [4]:
print(f'DataFrame shape: {user_data.shape}')
print(f'user_id nunique: {user_data.user_id.nunique()}')

DataFrame shape: (163205, 8)
user_id nunique: 163205


In [ ]:
post_text_df = pd.read_sql('SELECT * FROM public.post_text_df;', con=engine) 

,post_id,text,topic
0,1,UK economy facing major risks\n\nThe UK manufa...,business
1,2,Aids and climate top Davos agenda\n\nClimate c...,business
2,3,Asian quake hits European shares\n\nShares in ...,business
3,4,India power shares jump on debut\n\nShares in ...,business
4,5,Lacroix label bought by US firm\n\nLuxury good...,business


In [6]:
print(f'DataFrame shape: {post_text_df.shape}')
print(f'post_id nunique: {post_text_df.post_id.nunique()}')

DataFrame shape: (7023, 3)
post_id nunique: 7023


In [ ]:
source_feed_limit = 1000000
feed_data = pd.read_sql(f'SELECT * FROM public.feed_data LIMIT {source_feed_limit};', con=engine) 

,timestamp,user_id,post_id,action,target
0,2021-10-08 14:16:41,144623,3846,view,0
1,2021-10-08 14:19:19,144623,2735,view,0
2,2021-10-08 14:21:11,144623,3477,view,0
3,2021-10-08 14:22:07,144623,4526,view,0
4,2021-10-08 14:22:21,144623,5736,view,0


In [8]:
post_feed_df = post_text_df.merge(feed_data, on='post_id')
result_df = user_data.merge(post_feed_df, on='user_id')

In [9]:
result_df.isna().sum()

user_id      0
gender       0
age          0
country      0
city         0
exp_group    0
os           0
source       0
post_id      0
text         0
topic        0
timestamp    0
action       0
target       0
dtype: int64

In [11]:
print(f'DataFrame shape: {result_df.shape}')
print(f'post_id nunique: {result_df.post_id.nunique()}')
print(f'user_id nunique: {result_df.user_id.nunique()}')

DataFrame shape: (1000000, 14)
post_id nunique: 6831
user_id nunique: 2124


In [12]:
df = result_df.copy()

In [ ]:
df.user_id = df.user_id.astype(str)
df.post_id = df.post_id.astype(str)
df = df.sort_values('timestamp')

In [ ]:
df['text_len'] = df['text'].apply(len) # новый столбец с длиной строк из 'text'

In [ ]:
### Разделение на трейн-тест

train = df.iloc[:-200000].copy()
test = df.iloc[200000:].copy()

In [ ]:
### сколько постов просмотрел юзер за тренировочный период
user_count_posts = train.groupby('user_id').size()
train['user_view_posts'] = train['user_id'].map(user_count_posts)

,user_id,gender,age,country,city,exp_group,os,source,post_id,text,topic,timestamp,action,target,text_len,user_view_posts
243418,60373,1,20,Russia,Saint Petersburg,4,iOS,ads,1153,Visa row mandarin made Sir John\n\nThe top civ...,politics,2021-10-01 06:02:14,view,1,2537,494
243419,60373,1,20,Russia,Saint Petersburg,4,iOS,ads,1153,Visa row mandarin made Sir John\n\nThe top civ...,politics,2021-10-01 06:02:29,like,0,2537,494
243623,60373,1,20,Russia,Saint Petersburg,4,iOS,ads,3494,It’s wild how many people think they’re person...,covid,2021-10-01 06:02:31,view,0,140,494
243569,60373,1,20,Russia,Saint Petersburg,4,iOS,ads,2130,Iran jails blogger for 14 years\n\nAn Iranian ...,tech,2021-10-01 06:03:15,view,0,2535,494
243919,60373,1,20,Russia,Saint Petersburg,4,iOS,ads,7262,Ko to tamo peva is the best comedy of all time...,movie,2021-10-01 06:05:39,view,0,618,494


In [ ]:
### Среднее количество просмотров всех юзеров

overall_views_mean = user_count_posts.mean()


test['user_view_posts'] = (
    test['user_id']
    .map(user_count_posts)
    .fillna(overall_views_mean)
)

train = train.drop(['user_id', 'post_id', 'action','timestamp', 'text'], axis=1)
test = test.drop(['user_id', 'post_id', 'action', 'timestamp', 'text'], axis=1)

In [ ]:
X_train = train.drop('target', axis=1)
X_test = test.drop('target', axis=1)
y_train = train['target']
y_test = test['target']

,gender,age,country,city,exp_group,os,source,topic,text_len,user_view_posts
243418,1,20,Russia,Saint Petersburg,4,iOS,ads,politics,2537,494
243419,1,20,Russia,Saint Petersburg,4,iOS,ads,politics,2537,494
243623,1,20,Russia,Saint Petersburg,4,iOS,ads,covid,140,494
243569,1,20,Russia,Saint Petersburg,4,iOS,ads,tech,2535,494
243919,1,20,Russia,Saint Petersburg,4,iOS,ads,movie,618,494


In [ ]:
from catboost import CatBoostClassifier

catboost = CatBoostClassifier()

catboost.fit(X_train,
             y_train,
             cat_features=['gender', 'country', 'city', 'os', 'source', 'topic'],
)

In [28]:
y_pred = catboost.predict(X_test)

In [29]:
print(classification_report(y_test, y_pred, digits=4))

              precision    recall  f1-score   support

           0     0.8884    1.0000    0.9409    710733
           1     0.0000    0.0000    0.0000     89267

    accuracy                         0.8884    800000
   macro avg     0.4442    0.5000    0.4705    800000
weighted avg     0.7893    0.8884    0.8359    800000



In [30]:
print(f1_score(y_test, y_pred, average='weighted'))

0.8359154297740685


In [ ]:
catboost.save_model('control',
                    format="cbm")